In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [13]:
class LQRMonteCarloConstant:
    """
    Monte Carlo simulation of an LQR system with constant control alpha = [1,1]^T.
    Supports explicit and implicit Euler schemes.
    """

    def __init__(self, H, M, C, D, R, sigma, T, device="cpu"):
        self.device = device
        self.H = torch.tensor(H, dtype=torch.float32, device=device)
        self.M = torch.tensor(M, dtype=torch.float32, device=device)
        self.C = torch.tensor(C, dtype=torch.float32, device=device)
        self.D = torch.tensor(D, dtype=torch.float32, device=device)
        self.R = torch.tensor(R, dtype=torch.float32, device=device)
        self.sig = torch.tensor(sigma, dtype=torch.float32, device=device)
        self.T = T
        self.alpha_const = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device)

    def simulate_explicit(self, t0, x0, n_steps, n_mc, seed=None):
        if seed is not None:
            np.random.seed(seed)
            torch.manual_seed(seed)

        tau = (self.T - t0) / n_steps
        tau_sqrt = torch.sqrt(torch.tensor(tau, device=self.device))
        X = torch.tensor(x0, dtype=torch.float32, device=self.device).repeat(n_mc, 1)
        cost = torch.zeros(n_mc, device=self.device)

        for _ in range(n_steps):
            # drift term: constant control
            drift = tau * (X @ self.H.T) + tau * (self.M @ self.alpha_const).unsqueeze(0)

            # running cost: x' C x + alpha' D alpha
            state_cost = torch.sum((X @ self.C) * X, dim=1)
            ctrl_cost = torch.sum((self.alpha_const @ self.D) * self.alpha_const)
            cost += tau * (state_cost + ctrl_cost)

            # Euler step
            dW = torch.randn(n_mc, 2, device=self.device) * tau_sqrt
            X = X + drift + (self.sig @ dW.T).T

        # terminal cost
        cost += torch.sum((X @ self.R) * X, dim=1)
        return cost.cpu().numpy()

    def simulate_implicit(self, t0, x0, n_steps, n_mc, seed=None):
        if seed is not None:
            np.random.seed(seed)
            torch.manual_seed(seed)

        tau = (self.T - t0) / n_steps
        tau_sqrt = torch.sqrt(torch.tensor(tau, device=self.device))
        X = torch.tensor(x0, dtype=torch.float32, device=self.device).repeat(n_mc, 1)
        cost = torch.zeros(n_mc, device=self.device)
        I = torch.eye(2, dtype=torch.float32, device=self.device)

        for _ in range(n_steps):
            A = self.H
            sys_mat = I - tau * A

            dW = torch.randn(n_mc, 2, device=self.device) * tau_sqrt
            rhs = X + tau * (self.M @ self.alpha_const).unsqueeze(0) + (self.sig @ dW.T).T
            X = torch.linalg.solve(sys_mat, rhs.T).T

            # running cost
            state_cost = torch.sum((X @ self.C) * X, dim=1)
            ctrl_cost = torch.sum((self.alpha_const @ self.D) * self.alpha_const)
            cost += tau * (state_cost + ctrl_cost)

        # terminal cost
        cost += torch.sum((X @ self.R) * X, dim=1)
        return cost.cpu().numpy()

    def estimate_value(self, t0, x0, n_steps, n_mc, method="explicit", seed=None):
        if method == "explicit":
            costs = self.simulate_explicit(t0, x0, n_steps, n_mc, seed)
        else:
            costs = self.simulate_implicit(t0, x0, n_steps, n_mc, seed)
        return np.mean(costs), np.std(costs) / np.sqrt(n_mc)


class DGMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers):
        super(DGMNet, self).__init__()

        self.input = nn.Linear(input_dim, hidden_dim)
        self.hidden = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.output = nn.Linear(hidden_dim, 1)

    def activation(self, x):
        # Swish / SiLU activation (smooth and good for PDEs)
        return x * torch.sigmoid(x)

    def forward(self, t, x):
        # concatenate time and space
        inp = torch.cat([t, x], dim=1)

        h = self.activation(self.input(inp))
        for layer in self.hidden:
            h = self.activation(layer(h))

        return self.output(h)

class DeepGalerkinSolver:
    def __init__(self, H, M, C, D, R, sigma, alpha,
                 T,
                 hidden_dim,
                 n_layers,
                 device):

        self.device = device
        self.T = T

        # Convert matrices to tensors
        self.H = torch.tensor(H, dtype=torch.float32, device=self.device)
        self.M = torch.tensor(M, dtype=torch.float32, device=self.device)
        self.C = torch.tensor(C, dtype=torch.float32, device=self.device)
        self.D = torch.tensor(D, dtype=torch.float32, device=self.device)
        self.R = torch.tensor(R, dtype=torch.float32, device=self.device)
        self.sigma = torch.tensor(sigma, dtype=torch.float32, device=self.device)
        self.alpha = torch.tensor(alpha, dtype=torch.float32, device=self.device)

        self._validate_positive_definite(self.C, "C")
        self._validate_positive_definite(self.R, "R")
        self._validate_strictly_positive_definite(self.D, "D")

        self.dim = self.H.shape[0]

        self.net = DGMNet(self.dim + 1, hidden_dim, n_layers).to(self.device)

        self.optimizer = optim.Adam(self.net.parameters(), lr=1e-3)

    def _validate_positive_definite(self, A, name):
        """
        Checks if A is symmetric positive definite.
        Raises ValueError if not.
        """

        # Check symmetry
        if not torch.allclose(A, A.T, atol=1e-6):
            raise ValueError(f"Matrix {name} is not symmetric.")

        # Cholesky test
        try:
            torch.linalg.cholesky(A)
        except RuntimeError:
            raise ValueError(f"Matrix {name} is not positive definite.")

    def _validate_strictly_positive_definite(self, A, name):
        """
        Checks if A is strictly positive definite
        (all eigenvalues strictly > 0).
        """

        # Check symmetry
        if not torch.allclose(A, A.T, atol=1e-6):
            raise ValueError(f"Matrix {name} is not symmetric.")

        # Eigenvalue check
        eigenvalues = torch.linalg.eigvalsh(A)
        min_eig = torch.min(eigenvalues)

        if min_eig <= 0:
            raise ValueError(
                f"Matrix {name} is not strictly positive definite. "
                f"Minimum eigenvalue = {min_eig.item():.6e}"
            )

    def pde_residual(self, t, x):
        t.requires_grad_(True)
        x.requires_grad_(True)

        u = self.net(t, x)

        # Time derivative
        u_t = torch.autograd.grad(
            u, t,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        # First spatial gradient
        grad_u = torch.autograd.grad(
            u, x,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        # Hessian (loop over dimensions)
        hessian = []
        for i in range(self.dim):
            grad_i = torch.autograd.grad(
                grad_u[:, i],
                x,
                grad_outputs=torch.ones_like(grad_u[:, i]),
                create_graph=True
            )[0]
            hessian.append(grad_i)

        hessian = torch.stack(hessian, dim=2)  # shape: (batch, dim, dim)

        # Diffusion term: 1/2 tr(σσ^T Hessian)
        sigma_sigmaT = self.sigma @ self.sigma.T
        diffusion = 0.5 * torch.einsum("ij,bji->b", sigma_sigmaT, hessian)

        # Drift gradient terms
        Hx = x @ self.H.T
        Malpha = self.alpha @ self.M.T

        drift_term = (grad_u * (Hx + Malpha)).sum(dim=1)

        # Quadratic source terms
        quad_x = torch.einsum("bi,ij,bj->b", x, self.C, x)
        quad_alpha = self.alpha @ self.D @ self.alpha

        residual = (
            u_t.squeeze()
            + diffusion
            + drift_term
            + quad_x
            + quad_alpha
        )

        return residual

    def loss(self, batch_size):

        # Sample interior points
        t = torch.rand(batch_size, 1, device=self.device) * self.T
        x = torch.randn(batch_size, self.dim, device=self.device)

        res = self.pde_residual(t, x)
        pde_loss = torch.mean(res**2)

        # Terminal condition
        x_T = torch.randn(batch_size, self.dim, device=self.device)
        t_T = torch.ones(batch_size, 1, device=self.device) * self.T

        u_T = self.net(t_T, x_T)
        target = torch.einsum("bi,ij,bj->b", x_T, self.R, x_T)

        terminal_loss = torch.mean((u_T.squeeze() - target)**2)

        return pde_loss + terminal_loss

    def evaluate(self, t0, x0):
      """
      Evaluate the current network approximation at given (t0, x0).

      Parameters:
          t0 : float or tensor of shape (batch, 1)
              Initial time
          x0 : np.ndarray or tensor of shape (batch, dim)
              Initial state(s)
          n_steps, n_mc : optional, included for API compatibility with train()
      Returns:
          scalar float : predicted value at (t0, x0)
      """

      # Convert to tensors
      if not torch.is_tensor(x0):
          x0 = torch.tensor(x0, dtype=torch.float32, device=self.device)
      if not torch.is_tensor(t0):
          t0 = torch.tensor(t0, dtype=torch.float32, device=self.device)

      # Ensure proper shapes
      if x0.ndim == 1:
          x0 = x0.unsqueeze(0)  # shape (1, dim)
      if t0.ndim == 0:
          t0 = t0.unsqueeze(0).unsqueeze(1)  # shape (1,1)
      elif t0.ndim == 1:
          t0 = t0.unsqueeze(1)  # shape (batch,1)

      # Forward pass through the network
      with torch.no_grad():
          u = self.net(t0, x0)

      # Return a scalar (mean if batch>1)
      return u.mean().item()

    def train(self, mc_mean, t0, x0, epochs=5000, batch_size=256, method='explicit'):
      """
      Train the model and record loss history and error against a given Monte Carlo mean.

      Parameters:
          mc_mean : float
              Precomputed Monte Carlo reference value.
          epochs : number of training iterations
          batch_size : batch size for loss evaluation
          method : 'explicit' or 'implicit' (optional, in case needed for evaluation)
      Returns:
          loss_history : list of training losses
          error_history : list of absolute error vs Monte Carlo
      """

      self.loss_history = []
      self.error_history = []

      for epoch in range(epochs):
          self.optimizer.zero_grad()

          # Compute loss on batch
          loss = self.loss(batch_size)
          loss.backward()

          self.optimizer.step()

          # Record loss
          self.loss_history.append(loss.item())

          # Compute current model value and error vs precomputed MC mean
          current_value = self.evaluate(t0, x0)  # should return scalar
          error = abs(current_value - mc_mean)
          self.error_history.append(error)

          if epoch % 250 == 249:
              print(f"Epoch {epoch}, Loss: {loss.item():.6f}, Error vs MC: {error:.6f}")

      return self.loss_history, self.error_history

if __name__ == "__main__":
    # problem matrices (same as coursework spec)
    H = np.array([[0.5, 0.1], [0.1, 0.3]])
    M = np.array([[1.0, 0.5], [0.0, 1.0]])
    C = np.array([[2.0, 0.5], [0.5, 1.0]])
    D = np.array([[2.0, 0.0], [0.0, 1.0]])
    R = np.array([[1.0, 0.0], [0.0, 2.0]])
    sigma = np.array([[0.3, 0.1], [0.0, 0.2]])
    alpha = np.array([1,1])
    T = 1.0
    # Hyperparameters n_layers and hidden_dim
    n_layers = 4
    hidden_dim = 64

    epochs = 10000

    device = "cuda" if torch.cuda.is_available() else "cpu"
    MC_Sol = LQRMonteCarloConstant(H, M, C, D, R, sigma, T, device=device)
    t0=0
    x0 = np.array([0.0, 0.0])
    mean, stderr = MC_Sol.estimate_value(t0, x0, n_steps=100000, n_mc=100000, method="explicit")
    print("Mean:", mean, "StdErr:", stderr)

    solver = DeepGalerkinSolver(
    H=H,
    M=M,
    C=C,
    D=D,
    R=R,
    sigma=sigma,
    alpha=alpha,
    T=T,
    hidden_dim=hidden_dim,
    n_layers=n_layers,
    device="cuda" if torch.cuda.is_available() else "cpu"
    )
    solver.train(mc_mean=mean, epochs=epochs, t0=t0, x0=x0)


C:\Users\bpjam\AppData\Local\Temp\ipykernel_23364\3670009878.py:60: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  X = X + tau * (X @ H.T) + tau * (M @ alpha_const).unsqueeze(0) + (self.sigma @ dW.T).T


TypeError: unsupported operand type(s) for @: 'numpy.ndarray' and 'Tensor'

In [ ]:
    plt.figure()
    plt.plot(range(1, len(solver.loss_history) + 1), np.log(solver.loss_history),
             linewidth=0.2)
    plt.xlabel("Epoch")
    plt.ylabel("Log of Loss")
    plt.title("Log of Training Loss vs Epochs")
    plt.show()
    plt.savefig("GDMplotloss.png")

      # Plot log of error against Monte Carlo
    plt.figure()
    plt.plot(range(1, len(solver.error_history) + 1), np.log(solver.error_history),
             linewidth=0.2)

    # Noise floor (95% CI threshold)
    threshold = 1.96 * stderr
    plt.axhline(y=np.log(threshold),
            linestyle='--',
            color='red',
            label="95% MC Noise Level")

    plt.xlabel("Epoch")
    plt.ylabel("Log of Absolute Error vs MC")
    plt.title("Log of Error vs Monte Carlo Solution vs Epochs")
    plt.show()
    plt.savefig("DGMplot_error_vs_MC.png")